# Remove rooftop arrays by building intersection
NOTE: Requries BigPanelGEE.yml environment

**Use Google Earth Engine (GEE) to Remove Rooftop Arrays from Existing Solar Array Datasets**
* Inputs: *existingSolarArrayShapes.shp* local shape file and asset upload to GEE.
* Uses GEE to pull in [USA Structures](https://gee-community-catalog.org/projects/usa_structures/?h=ornl) dataset, calcualte the intersection with our newly-compiled solar array dataset, and remove arrays that are likely rooftop mounted. 
* Output: Ground mounted solar array dataset.

## Import Libraries

In [1]:
# Import libraries
import numpy as np
import pandas as pd
import geopandas as gpd
import os 
import re
import ee
from shapely.ops import unary_union
# import geemap

# Import gmseusUtils
import gmseusUtils as gu

## Initialize GEE

In [2]:
# Trigger the GEE authentication
ee.Authenticate()

# Initialize the cloud project
ee.Initialize(project='ee-stidjaco')

## Set paths and variables

In [8]:
# Set folder paths
wd = r'S:\Users\stidjaco\R_files\BigPanel'
downloaded_path = os.path.join(wd, r'Data\Downloaded')
derived_path = os.path.join(wd, r'Data\Derived')
derivedTemp_path = os.path.join(derived_path, r'intermediateProducts')

# Set country name
countryName = 'USA'

# Set solar array and panel shapefile locations (asset and local)
arraysLocalPath = os.path.join(derivedTemp_path, r'GMSEUS_Arrays_selectedBoundaries.shp') # We create this file in this script4
panelsLocalPath = os.path.join(derivedTemp_path, r'GMSEUS_Panels_ExistingAndNAIP.shp') # We create this file in this script4
arraysAssetPath = r'projects/ee-stidjaco/assets/BigPanel/GMSEUS_Arrays_selectedBoundaries' # We upload this file in script4
existingRooftopArraysPath = os.path.join(derivedTemp_path, r'existingRooftopArrayShapes.shp') # GM-SEUS-generated rooftop arrays - script2

# GM-SEUS ground-mounted paths
gmseusArraysGroundMountedPath = os.path.join(derivedTemp_path, r'GMSEUS_Arrays_GroundMounted.shp')
gmseusPanelsGroundMountedPath = os.path.join(derivedTemp_path, r'GMSEUS_Panels_GroundMounted.shp')

# Load the config from the text file
config = gu.load_config()

# Set threshold for building area contained within array inferring rooftop
build_threshold = config['build_threshold'] # 50% of building area contained within array

# Other variables
currentVersion = config['currentVersion'] # version of the dataset
gee_crs = config['gee_crs'] # native projection of Google Earth Engine exports
minPanelRowArea = config['minPanelRowArea'] # minimum area of panel to be considered
toCRS = config['to_crs']  # EPSG:6350 NAD83 (2011)

# Append toCRS with the EPSG prefix for use in GeoPandas
toCRS = f'EPSG:{toCRS}'

# Set error margin
errMargin = ee.ErrorMargin(10)

# Set local and global arrayID columns
arrayIDcol = 'subArrID'
arrayIDcolGlob = 'tmpArrID'

## Acquire Buildings Asset

In [5]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Open Building Maps Building Dataset

# Get building database path
buildingAssetBasePathOBM = r"projects/sat-io/open-datasets/OPEN-BUILDING-MAPS/tiles"
buildingAssetTiles = r"projects/sat-io/open-datasets/OPEN-BUILDING-MAPS/open_buildings_grid"

# Tile grid
buildingTiles = ee.FeatureCollection(buildingAssetTiles)

# Get array centroids and bounds
arraysAsset = ee.FeatureCollection(arraysAssetPath)
arraysCentroids = arraysAsset.map(lambda f: f.setGeometry(f.geometry().centroid(1)))
arraysBounds = arraysCentroids.geometry().bounds(1)

# Get the filenames of only those tiles (client-side list of strings)
tile_filenames = buildingTiles.filterBounds(arraysBounds).aggregate_array('filename').getInfo()

# Extract tile keys (numeric parts) from filenames
tile_keys = []
for fn in tile_filenames:
    m = re.search(r'(\d+)', fn)
    if m:
        tile_keys.append(m.group(1))  # e.g. "12345"

# Turn them into expected asset names, e.g. "building_12345"
tile_asset_names = {f'building_{k}' for k in tile_keys}

# List *all* assets in the OBM tiles folder (metadata only, not geometries)
asset_list = ee.data.listAssets({'parent': buildingAssetBasePathOBM}).get('assets', [])

# Convert only relevant assets to FeatureCollections
def asset_to_fc(asset):
    return ee.FeatureCollection(asset['name'])
building_fc_list = [
    asset_to_fc(a)
    for a in asset_list
    if a['name'].split('/')[-1] in tile_asset_names
]

# Merge all selected tile FeatureCollections into one big FC
buildingsAsset = ee.FeatureCollection(building_fc_list).flatten()

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Using Microsoft VIDA Building Dataset

# # Get building database path
# buildingAssetPathVIDA = r"projects/sat-io/open-datasets/VIDA_COMBINED/USA"
# buildingsAsset = ee.FeatureCollection(buildingAssetPathVIDA)

## Get solar array intersections with buildings and save as pandas df

In [6]:
# ~1500 minutes

# Call asset arrays
arraysAsset = ee.FeatureCollection(arraysAssetPath)

# Create a temporary folder within the derivedTemp_path to store the results
rooftopResultsPath = os.path.join(derivedTemp_path, 'rooftopResults')
gu.checkFolder(rooftopResultsPath)

# Reorder arraysAsset by increasing latitude then increasing longitude to make chunking more spatially coherent
def getLatLon(feature):
    centroid = feature.geometry().centroid()
    lat = centroid.coordinates().get(1)
    lon = centroid.coordinates().get(0)
    return feature.set({'lat': lat, 'lon': lon})
arraysAsset = arraysAsset.map(getLatLon)
arraysAsset = arraysAsset.sort('lat').sort('lon')

# Get assed ID list
assetIDList = arraysAsset.aggregate_array(arrayIDcol).getInfo()

# Function to calculate the intersection and proportional area
def calculate_intersection_area(array):
    # Get the array geometry
    aoiTemp = array.geometry()

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Select buldings dataset, and get local buildings

    # Open Building Maps dataset
    # Get local buildings with an area less than global surface area (Largest building footprint ~1e6 sqm)
    localBuildings = ee.Geometry.MultiPolygon(buildingsAsset
        .filterBounds(aoiTemp)
        .map(lambda f: f.set({'footprintArea': f.geometry().area(errMargin)}))
        .filter(ee.Filter.lt('footprintArea', 10e6))
        .geometry().geometries())

    # Microsoft VIDA buildings dataset
    # Filter the building vectors to the bounds of the aoi. 
    # For this Google Buildings Dataset, this always results in 29 features, only the last of which is the correct shape. We have not ascertained why this is the case. 
    # When we filter and acquire multipolygon geometries (if more than one building intersects), the correct shapes are index 29:numFeatures. 
    # localBuildings = buildingsAsset.filterBounds(aoiTemp).geometry().geometries() 
    # actualFootprintStart = 29 # First building footprint index for the Google Buildings Dataset. 
    # numGeometries = localBuildings.size() # Get total number of polyigons
    # validGeometries = localBuildings.slice(actualFootprintStart, numGeometries) # Slice the geometries from index 29 to the end
    # localBuildings = ee.Geometry.MultiPolygon(validGeometries) # Convert the list of valid geometries back into a MultiPolygon

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ 

    # Now, acquire the intersecting area of roof and solar and get the rooftop proportion
    intersectionArea = aoiTemp.intersection(localBuildings, errMargin).area(errMargin)
    rooftopAreaProp = intersectionArea.divide(aoiTemp.area(errMargin)).multiply(100).toInt()

    # Set as new attribute and return the feature
    return array.set({'roofProp': rooftopAreaProp})

# Break the assetIDList into equal chunks smaller than 5000 to overcome GEE memory limitations. Then, for each chunk, get the corresponding feature collection and apply the calculate_intersection_area function to each feature. Append the results to the lists.
chunkSize = 10 # This used to be 4999 to prevent memory issues using ORNL structures dataset, but with the google dataset, we needed multiple exports of csv to overcome memory issues.
chunks = [assetIDList[i:i + chunkSize] for i in range(0, len(assetIDList), chunkSize)]

# For each chunk, get the corresponding feature collection and apply the calculate_intersection_area function to each feature. Append the results to the lists.
processed = 0
last_pct = -1
total = len(assetIDList)
for chunk in chunks:
    
    # Initialize lists
    id_list = []
    roofProp_list = []

    # Get the feature collection for the chunk
    arraysAssetChunk = arraysAsset.filter(ee.Filter.inList(arrayIDcol, ee.List(chunk)))

    # Apply the function to each feature in the arrays collection, and append the results to the lists
    arrays_with_rooftop = arraysAssetChunk.map(calculate_intersection_area)
    for feature in arrays_with_rooftop.getInfo()['features']:
        id_list.append(feature['properties'][arrayIDcol])
        roofProp_list.append(feature['properties']['roofProp'])

    # Create a dictionary and convert to a dataframe
    data = {arrayIDcol: id_list, 'roofProp': roofProp_list}
    arraysRooftopDf = pd.DataFrame(data)

    # Print progress through chunks (only at % integer intervals)
    processed += len(chunk)
    pct = (processed * 100) // total  # integer percent
    if pct > last_pct:
        print(f"Processed {processed} of {total} arrays ({pct}% complete)")
        last_pct = pct
        
    # Export the result to CSV
    arraysRooftopDf.to_csv(os.path.join(rooftopResultsPath, 'arraysRooftopProp'+str(chunk[0])+'.csv'), index=False)

# Call in all the csv files and concatenate them into one dataframe from the rooftopResultsPath
arraysRooftopDf = pd.concat([pd.read_csv(os.path.join(rooftopResultsPath, f)) for f in os.listdir(rooftopResultsPath)], ignore_index=True)

# Check to ensure indexing logic is correct
print("Number of arrays assessed: ", len(arraysRooftopDf))
print("Correct total number of arrays: ", arraysAsset.size().getInfo())

# Export dataframe to derivedTemp_path
arraysRooftopDf.to_csv(os.path.join(derivedTemp_path, 'arraysRooftopProp.csv'), index=False)

Processed 10 of 31456 arrays (0% complete)
Processed 320 of 31456 arrays (1% complete)
Processed 630 of 31456 arrays (2% complete)
Processed 950 of 31456 arrays (3% complete)
Processed 1260 of 31456 arrays (4% complete)
Processed 1580 of 31456 arrays (5% complete)
Processed 1890 of 31456 arrays (6% complete)
Processed 2210 of 31456 arrays (7% complete)
Processed 2520 of 31456 arrays (8% complete)
Processed 2840 of 31456 arrays (9% complete)
Processed 3150 of 31456 arrays (10% complete)
Processed 3470 of 31456 arrays (11% complete)
Processed 3780 of 31456 arrays (12% complete)
Processed 4090 of 31456 arrays (13% complete)
Processed 4410 of 31456 arrays (14% complete)
Processed 4720 of 31456 arrays (15% complete)
Processed 5040 of 31456 arrays (16% complete)
Processed 5350 of 31456 arrays (17% complete)
Processed 5670 of 31456 arrays (18% complete)
Processed 5980 of 31456 arrays (19% complete)
Processed 6300 of 31456 arrays (20% complete)
Processed 6610 of 31456 arrays (21% complete)
Pro

## Remove buildings from existing and digitized solar array and panel databases

### Arrays

In [9]:
# Call existing ground-mounted and rooftop arrays
arraysLocal = gpd.read_file(arraysLocalPath)
rooftopArraysLocal = gpd.read_file(existingRooftopArraysPath)

# Print length of local arrays
print("Original number of arrays: ", len(arraysLocal))

# Call the CSV
arraysRooftopDf = pd.read_csv(os.path.join(derivedTemp_path, 'arraysRooftopProp.csv'))

# Merge the dataframes on a common identifier
mergedArrays = arraysLocal.merge(arraysRooftopDf[[arrayIDcol, 'roofProp']], on=arrayIDcol)

# Drop area column, since we have exploded multipolygons into single polygons and area values no longer represent actual array area
mergedArrays = mergedArrays.drop(columns=['area'])
mergedArrays['area'] = mergedArrays.geometry.area.round(0) # We will again calculate area after re-merging by installation year

# Save rooftop arays in mergedArrays, select for initial columns, check for overlap with rooftopArraysLocal, and append
rooftopArraysBuildThresh = mergedArrays[mergedArrays['roofProp'] > build_threshold]
rooftopArraysBuildThresh = rooftopArraysBuildThresh[rooftopArraysLocal.columns.tolist()]
rooftopArrays_fromGMSEUS = gu.preferentialSpatialFilter([rooftopArraysBuildThresh, rooftopArraysLocal])

# Add a unique roofArrID to rooftopArrays_fromGMSEUS
rooftopArrays_fromGMSEUS['roofArrID'] = range(1, len(rooftopArrays_fromGMSEUS) + 1)

# Drop mergedArrays with a rooftopProp below the threshold
mergedArrays = mergedArrays[mergedArrays['roofProp'] <= build_threshold]
mergedArrays = mergedArrays.reset_index(drop=True)

# Drop the temporary ID column, and add a new tempArrID column
mergedArrays = mergedArrays.drop(columns=[arrayIDcol])
mergedArrays[arrayIDcol] = range(1, len(mergedArrays) + 1)

# Print number of remaning arrays and total area of ground-mounted arrays
print("Number of arrays after filtering: ", len(mergedArrays))
print(f"Total area of ground-mounted arrays: {mergedArrays['area'].sum() / 1e6:.2f}")

# Print the number of rooftop arrays removed and area of rooftop arrays removed, then the total rooftop arrays and area determined by GMSEUS processess
print("Number of rooftop arrays removed: ", len(rooftopArraysBuildThresh))
print(f"Total area of rooftop arrays removed: {rooftopArraysBuildThresh['area'].sum() / 1e6:.2f}")
print("Total number of rooftop arrays determined by GMSEUS processes: ", len(rooftopArrays_fromGMSEUS))
print(f"Total area of rooftop arrays determined by GMSEUS processes: {rooftopArrays_fromGMSEUS['area'].sum() / 1e6:.2f}")

# Export the merged ground-mounted arrays to shapefile
mergedArrays.to_file(gmseusArraysGroundMountedPath, driver='ESRI Shapefile')

# Export the rooftop arrays to shapefile, geopackage, and csv
rooftopArrays_fromGMSEUS.to_file(os.path.join(derived_path, f'RooftopArraysFromGMSEUS_{currentVersion}.shp'), driver='ESRI Shapefile')

Original number of arrays:  31456
Number of arrays after filtering:  27828
Total area of ground-mounted arrays: 3819.24
Number of rooftop arrays removed:  3628
Total area of rooftop arrays removed: 22.25
Total number of rooftop arrays determined by GMSEUS processes:  5822
Total area of rooftop arrays determined by GMSEUS processes: 46.00


### Panels

In [ ]:
# Re-call GM-SEUS initial arrays and call the panel-row data (both are already in the correct projection)
gmseusArrays = gpd.read_file(gmseusArraysGroundMountedPath)
panelsLocal = gpd.read_file(panelsLocalPath)

# Print the original number of panels
print("Original number of panels: ", len(panelsLocal))

# Spatially join gmseus arrays to panels, copy the array id column to the panels, and drop the index columns. 
panelsLocal = gpd.sjoin(panelsLocal, gmseusArrays[[arrayIDcol, 'geometry']], how='left', predicate='intersects')
panelsLocal = panelsLocal.reset_index(drop=True)
panelsLocal = panelsLocal.drop(columns=['index_left', 'index_right'], errors='ignore')

# Drop panels that do not have an arrayIDcol
gmPanelsLocal = panelsLocal.dropna(subset=[arrayIDcol])

# Print the number of panels after filtering
print("Number of panels after filtering: ", len(gmPanelsLocal))

# Print the number of rooftop panels removed
print("Number of rooftop panels removed: ", len(panelsLocal) - len(gmPanelsLocal))

# Print the total area of rooftop panels removed in square km
print(f"Total area of rooftop panels removed: {(panelsLocal['area'].sum() - gmPanelsLocal['area'].sum()) / 1e6:.2f}")

# Print the total area of ground-mounted panels in square km
print(f"Total area of ground-mounted panels: {gmPanelsLocal['area'].sum() / 1e6:.2f}")

# Print the number of unique initIDs in the panels
print("Number of GMSEUS arrays with existing panels: ", len(gmPanelsLocal[arrayIDcol].unique()))

# Drop arrayIDcol column
gmPanelsLocal = gmPanelsLocal.drop(columns=[arrayIDcol])

# Save a new panelID column that is the row index (after resetting the index)
gmPanelsLocal = gmPanelsLocal.reset_index(drop=True)
gmPanelsLocal['panelID'] = gmPanelsLocal.index

# Export the panels to shapefile
gmPanelsLocal.to_file(gmseusPanelsGroundMountedPath, driver='ESRI Shapefile')

Original number of panels:  3569615
Number of panels after filtering:  3547205
Number of rooftop panels removed:  23865
Total area of ground-mounted panels: 536.40
Number of GMSEUS arrays with existing panels:  18099
